## Northwestern unemployment bar chart, faceted by state

In [ ]:
bar_unemployment = alt.Chart(unemp_data).mark_bar().transform_calculate(
    owns_facility="datum.is_facility>0 ? 'Yes' : 'No'"
).encode(
    x=alt.X('geoname_abbrev', sort='-y').title('').axis(labelAngle=-70),
    y=alt.Y('unemp_total').title('Unemployment Rate'),
    facet=alt.Facet('owner_state_full').title(''),
    fill=alt.Fill('is_us_state').scale(range=region_colors).legend(None),
    stroke=alt.Stroke('owns_facility:N').scale(range=["#FFFFFFFF", "#000000"]).legend(
        title=['Do any nations on', 'this reservation', 'own gaming facilities?'],
        symbolFillColor=region_colors[0]
    ),
    strokeWidth=alt.StrokeWidth('owns_facility:N', ).scale(range=[0, 2]).legend(None)
).resolve_scale(x='independent')


bar_unemployment.properties(
    height=300,
    width=250,
    title=alt.Title(['Northwestern reservations exceed nearby state unemployment rates in 2023'],
                    anchor='start',
                    subtitle='State-wide unemployment rates are highlighted. Only X Washington reservations are shown out of 29 total.',
                    # align='center',
                    fontSize=22,
                    subtitleFontSize=15,
                    dx=25,
                    dy=-10)
).configure(
    font='Agency FB',
).configure_axis(
    labelFontSize=14,
    titleFontSize=16,
).configure_headerFacet(
    labelFontSize=18
).configure_legend(
    titleFontSize=15,
    titleAlign='center',
    labelFontSize=15,
    # labelAlign='left',
    symbolOffset=-20,
    # labelOffset=0,
    orient='right',
    # legendY=0.5,
    strokeColor='lightgrey',
    padding=5,
    # titlePadding=15
)#.save('./graphics/bar_state_facet_unemployment_vs_reservation.svg')
# Texture for state vs. nation, top X unemployment rates, facet by year?

# Encode color of reservations to whether owns facility or not?
# Encode outline to owns facility!!

### Unemployment with casino ownership encoded to outline dash

In [ ]:
unemp_data_mt_or = unemp_data[unemp_data['owner_state'].isin(['MT', 'OR'])]
unemp_data_wa = unemp_data[unemp_data['owner_state'].isin(['WA'])]

chart_width = 250
chart_height = 300
top_row_spacing = 200
def make_unemp_facet(df, is_wide=False):
    return (
        alt.Chart(df)
        .mark_bar(strokeWidth=2)
        .transform_calculate(
            owns_facility="datum.is_facility>0 ? 'Yes' : 'No'"
        )
        .encode(
            x=alt.X('geoname_abbrev', sort='-y', title='', axis=alt.Axis(labelAngle=-70)),
            y=alt.Y('unemp_total', title='Unemployment Rate'),
            fill=alt.Fill('is_us_state').scale(range=region_colors).legend(None),
            stroke=alt.Stroke('is_us_state').scale(
                    range=["#000000", "#000000"]
                ).legend(
                    title=['Do any nations in', 'this region', 'own gaming facilities?'],
                    symbolFillColor=region_colors[0]
                ),
            strokeDash=alt.StrokeDash('owns_facility:N').scale(range=[[1,0], [10,10]])
            # strokeWidth=alt.StrokeWidth('owns_facility:N').scale(range=[0, 2]).legend(None)
        )
    ).properties(
    width=chart_width * 2**is_wide + is_wide*top_row_spacing,
    height=chart_height
)

chart_mt_or = make_unemp_facet(unemp_data_mt_or).facet(
    facet='owner_state_full:N',
    columns=2,
    spacing=top_row_spacing,
    title=''
).resolve_scale(x='independent')

chart_wa = make_unemp_facet(unemp_data_wa, is_wide=True).facet(
    facet='owner_state_full:N',
    columns=1,
    title=alt.Title('')
).resolve_scale(x='independent')

final_chart = (
    alt.vconcat(chart_mt_or, chart_wa)
    .resolve_scale(color='shared', stroke='shared')
    .properties(
        title=alt.TitleParams(
            ['Most northwestern reservations exceeded nearby state unemployment rates in 2023'],
            anchor='start',
            subtitle='State-wide unemployment rates are highlighted in orange.',
            fontSize=22,
            subtitleFontSize=15,
            dx=25,
            dy=-10
        )
    )
    .configure(font='Agency FB')
    .configure_axis(labelFontSize=14, titleFontSize=16)
    .configure_headerFacet(labelFontSize=18, title=None)
    .configure_legend(
        titleFontSize=15,
        titleAlign='center',
        labelFontSize=15,
        symbolOffset=-20,
        orient='none',
        direction='vertical',
        legendX=chart_width + top_row_spacing/5,
        legendY=chart_height + 12,
        strokeColor='lightgrey',
        padding=5
    )
)

final_chart